In [9]:
%pip install numpy pandas matplotlib pillow requests python-dotenv pydantic

print("Packages installed successfully.")

Note: you may need to restart the kernel to use updated packages.
Packages installed successfully.


## Shared Imports

Run this cell once before executing the step cells below.

In [14]:
import csv
import glob
import json
import os
import random
import sys
from datetime import datetime

import numpy as np
import pandas as pd

print("Shared imports loaded.")

Shared imports loaded.


## Data and Environment Overview

In this step we:
- read all `metadata.json` files from `environment_files`
- build a table for each game (`game_id`, tags, baseline actions)
- inspect tag distribution and save an overview CSV

In [15]:
def detect_workspace_root() -> str:
    """Find the project root in a simple, portable way."""
    override = os.environ.get("ARC_WORKSPACE")
    if override:
        override = os.path.abspath(os.path.expanduser(override))
        if os.path.isdir(os.path.join(override, "environment_files")):
            return override

    current = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, "environment_files")):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent

    # Common Kaggle competition mount point fallback.
    kaggle_root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
    if os.path.isdir(os.path.join(kaggle_root, "environment_files")):
        return kaggle_root

    raise FileNotFoundError(
        "Cannot find project root with 'environment_files'. "
        "Set ARC_WORKSPACE or run the notebook from the repository."
    )

WORKSPACE = detect_workspace_root()
ENV_DIR = os.path.join(WORKSPACE, "environment_files")

metadata_paths = sorted(glob.glob(os.path.join(ENV_DIR, "*", "*", "metadata.json")))
records = []

for metadata_path in metadata_paths:
    with open(metadata_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    baseline_actions = data.get("baseline_actions", [])
    tags = data.get("tags", [])

    env_py_candidates = sorted(
        glob.glob(os.path.join(os.path.dirname(metadata_path), "*.py"))
    )
    env_py_path = (
        os.path.relpath(env_py_candidates[0], WORKSPACE) if env_py_candidates else ""
    )

    records.append(
        {
            "game_id": data.get("game_id", ""),
            "title": data.get("title", ""),
            "tags": tags,
            "num_levels": len(baseline_actions),
            "baseline_min": min(baseline_actions) if baseline_actions else None,
            "baseline_max": max(baseline_actions) if baseline_actions else None,
            "baseline_avg": (
                round(sum(baseline_actions) / len(baseline_actions), 2)
                if baseline_actions
                else None
            ),
            "metadata_path": os.path.relpath(metadata_path, WORKSPACE),
            "env_py_path": env_py_path,
        }
    )

games_df = pd.DataFrame(records).sort_values("game_id").reset_index(drop=True)
view_df = games_df.copy()
view_df["tags"] = view_df["tags"].apply(lambda x: ", ".join(x) if x else "no_tags")

all_tags = []
for tags in games_df["tags"]:
    if tags:
        all_tags.extend(tags)
    else:
        all_tags.append("no_tags")

tag_df = (
    pd.Series(all_tags, name="tag")
    .value_counts()
    .rename_axis("tag")
    .reset_index(name="count")
)

print(f"Workspace root: {WORKSPACE}")
print(f"Total public games: {len(view_df)}")
print(f"Total public levels: {int(view_df['num_levels'].sum())}")
print(f"Average levels per game: {view_df['num_levels'].mean():.2f}")
print()
print("Tag distribution:")
display(tag_df)

print("\nPublic games overview:")
display(
    view_df[
        [
            "game_id",
            "title",
            "tags",
            "num_levels",
            "baseline_min",
            "baseline_max",
            "baseline_avg",
            "env_py_path",
        ]
    ]
)

overview_csv = os.path.join(WORKSPACE, "public_games_overview.csv")
view_df.to_csv(overview_csv, index=False)
print(f"\nOverview saved to {overview_csv}")

Workspace root: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3
Total public games: 25
Total public levels: 183
Average levels per game: 7.32

Tag distribution:


,tag,count
0,keyboard_click,13
1,click,7
2,keyboard,4
3,no_tags,1



Public games overview:


,game_id,title,tags,num_levels,baseline_min,baseline_max,baseline_avg,env_py_path
0,ar25-0c556536,AR25,keyboard_click,8,32,233,93.50,environment_files/ar25/0c556536/ar25.py
1,bp35-0a0ad940,BP35,keyboard_click,9,21,163,72.33,environment_files/bp35/0a0ad940/bp35.py
2,cd82-fb555c5d,CD82,keyboard_click,6,8,55,28.50,environment_files/cd82/fb555c5d/cd82.py
3,cn04-2fe56bfb,CN04,keyboard_click,6,29,300,131.50,environment_files/cn04/2fe56bfb/cn04.py
4,dc22-fdcac232,DC22,keyboard_click,6,59,578,204.67,environment_files/dc22/fdcac232/dc22.py
5,ft09-0d8bbf25,FT09,no_tags,6,12,65,34.67,environment_files/ft09/0d8bbf25/ft09.py
6,g50t-5849a774,G50T,keyboard,7,54,230,125.57,environment_files/g50t/5849a774/g50t.py
7,ka59-38d34dbb,KA59,keyboard_click,7,28,326,104.29,environment_files/ka59/38d34dbb/ka59.py
8,lf52-271a04aa,LF52,click,10,32,244,133.90,environment_files/lf52/271a04aa/lf52.py
9,lp85-305b61c3,LP85,click,8,16,159,48.50,environment_files/lp85/305b61c3/lp85.py



Overview saved to /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/public_games_overview.csv


## STEP 3: Experiment Infrastructure

In this step we create a minimal experiment framework:
- fixed random seed for reproducibility
- timestamped run directory
- standard files for config and metrics logging
- helper function to append metrics rows

In [17]:
def setup_experiment_environment(workspace_root: str, experiment_name: str, seed: int = 42):
    """Create run folders, set seeds, and prepare logging files."""
    random.seed(seed)
    np.random.seed(seed)

    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    run_name = f"{timestamp}_{experiment_name}"

    runs_dir = os.path.join(workspace_root, "runs")
    run_dir = os.path.join(runs_dir, run_name)
    metrics_dir = os.path.join(run_dir, "metrics")
    artifacts_dir = os.path.join(run_dir, "artifacts")

    os.makedirs(metrics_dir, exist_ok=True)
    os.makedirs(artifacts_dir, exist_ok=True)

    config = {
        "run_name": run_name,
        "experiment_name": experiment_name,
        "seed": seed,
        "workspace_root": workspace_root,
        "created_utc": datetime.utcnow().isoformat(),
        "python_version": sys.version.split()[0],
    }

    config_path = os.path.join(run_dir, "run_config.json")
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    metrics_csv = os.path.join(metrics_dir, "metrics.csv")
    with open(metrics_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "timestamp_utc",
            "step",
            "game_id",
            "metric_name",
            "metric_value",
            "notes",
        ])

    return {
        "run_dir": run_dir,
        "metrics_csv": metrics_csv,
        "config_path": config_path,
        "seed": seed,
    }


def log_metric(metrics_csv: str, step: str, game_id: str, metric_name: str, metric_value, notes: str = ""):
    """Append one metric record to metrics.csv."""
    with open(metrics_csv, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.utcnow().isoformat(),
            step,
            game_id,
            metric_name,
            metric_value,
            notes,
        ])


workspace_root = WORKSPACE if "WORKSPACE" in globals() else os.getcwd()
RUN_CONTEXT = setup_experiment_environment(
    workspace_root=workspace_root,
    experiment_name="arc3_baseline",
    seed=42,
)

# Example metric row to validate logging pipeline.
log_metric(
    metrics_csv=RUN_CONTEXT["metrics_csv"],
    step="step3",
    game_id="all",
    metric_name="infrastructure_ready",
    metric_value=1,
    notes="Experiment scaffolding initialized.",
)

print("Experiment infrastructure is ready.")
print(f"Run directory: {RUN_CONTEXT['run_dir']}")
print(f"Config file: {RUN_CONTEXT['config_path']}")
print(f"Metrics file: {RUN_CONTEXT['metrics_csv']}")

Experiment infrastructure is ready.
Run directory: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/20260423_112701_arc3_baseline
Config file: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/20260423_112701_arc3_baseline/run_config.json
Metrics file: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/20260423_112701_arc3_baseline/metrics/metrics.csv
